# S0.6 · R-B（个人信息出境标准合同路径）推演

S0.5 已经算出：R-A 的可用样本三档全部低于可验证下界，
且**任何达到可验证规模的双向 L3 方案都必然落入安全评估路径**。

因此本步的重点不是重复漏斗（结构性折损层与 R-A 相同），而是回答一个更有用的问题：

> **R-B 到底在什么条件下才用得上？**

方法：把三种 L3 形态（双向 / 单向 / 无 cn→hk 流动）与出境机制阈值套在一起，
看每种形态在不同人数下落到哪条路径。

阈值来自《促进和规范数据跨境流动规定》第五、七、八条，逐字原文见 `legal_references.md` 一之二节。

> ⚠️ 折损率与人群规模为显式假设；**阈值规则是法条原文，不是假设**。

In [1]:
import hashlib
import pathlib
import subprocess
import sys

import pandas as pd
import yaml

HASH_PREFIX_LEN = 12
PERCENT = 100

REPO_ROOT = pathlib.Path.cwd()
while not (REPO_ROOT / "AGENTS.md").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

CONFIG_PATH = REPO_ROOT / "modules/m0_compliance/configs/s0_6_route_b.yaml"
cfg = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
config_hash = hashlib.sha256(CONFIG_PATH.read_bytes()).hexdigest()[:HASH_PREFIX_LEN]
git_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()

print("config:", CONFIG_PATH.relative_to(REPO_ROOT), "sha256:" + config_hash)
print("seed:", cfg["seed"])
print("git:", git_sha)
print("step:", cfg["step_id"], "| 路线:", cfg["route"], cfg["route_name"])

config: modules/m0_compliance/configs/s0_6_route_b.yaml sha256:f753a85deddb
seed: 42
git: e09725a
step: S0.6 | 路线: R-B 个人信息出境标准合同路径（PIPL 第 38 条第（三）项）


In [2]:
from modules.m0_compliance.components.funnel import attrition_rate, funnel

rows = funnel(cfg["base_population"], cfg["stages"])
table = pd.DataFrame(rows)
table.to_csv(REPO_ROOT / "modules/m0_compliance/results/route_B_funnel.csv", index=False)

final = rows[-1]
print("R-B 路径下可用样本：low %d / mid %d / high %d 人"
      % (final["remaining_low"], final["remaining_mid"], final["remaining_high"]))
print("业务最小可行规模（S0.1）：%d 人" % cfg["minimum_viable_n_eff"])
print()
print("对照 R-A（S0.5）：low 3 / mid 226 / high 5814 人")
print("同意率上调后仍然三档不达标：",
      all(final["remaining_%s" % s] < cfg["minimum_viable_n_eff"] for s in ("low", "mid", "high")))

R-B 路径下可用样本：low 4 / mid 264 / high 6395 人
业务最小可行规模（S0.1）：30000 人

对照 R-A（S0.5）：low 3 / mid 226 / high 5814 人
同意率上调后仍然三档不达标： True


In [3]:
from modules.m0_compliance.components.export_mechanism import required_mechanism

thresholds = cfg["thresholds"]
scales = cfg["scale_points"]

records = []
for form in cfg["l3_forms"]:
    for n in scales:
        assets = form["cn_to_hk_assets"]
        count = n if assets else 0
        records.append({
            "L3 形态": form["id"],
            "cn→hk 提供": "、".join(assets) if assets else "无",
            "含敏感个人信息": "是" if form["cn_to_hk_sensitive"] else "否",
            "涉及人数": n,
            "所需机制": required_mechanism(count, form["cn_to_hk_sensitive"], thresholds),
        })

matrix = pd.DataFrame(records)
matrix.to_csv(REPO_ROOT / "modules/m0_compliance/results/export_mechanism_matrix.csv", index=False)
print(matrix.to_string(index=False))

L3 形态                 cn→hk 提供 含敏感个人信息    涉及人数            所需机制
  形态A CBA-002 标识符哈希、CBA-005 梯度       是    5000   标准合同或个人信息保护认证
  形态A CBA-002 标识符哈希、CBA-005 梯度       是   30000        数据出境安全评估
  形态A CBA-002 标识符哈希、CBA-005 梯度       是  150000        数据出境安全评估
  形态A CBA-002 标识符哈希、CBA-005 梯度       是 1200000        数据出境安全评估
  形态B            CBA-002 标识符哈希       否    5000  豁免（无需申报/订立/认证）
  形态B            CBA-002 标识符哈希       否   30000  豁免（无需申报/订立/认证）
  形态B            CBA-002 标识符哈希       否  150000   标准合同或个人信息保护认证
  形态B            CBA-002 标识符哈希       否 1200000        数据出境安全评估
  形态C                        无       否    5000 不适用（该方向不提供个人信息）
  形态C                        无       否   30000 不适用（该方向不提供个人信息）
  形态C                        无       否  150000 不适用（该方向不提供个人信息）
  形态C                        无       否 1200000 不适用（该方向不提供个人信息）


In [4]:
viable = cfg["minimum_viable_n_eff"]
print("在可验证规模 N_eff = %d 人这一档上，三种形态分别落到：" % viable)
for form in cfg["l3_forms"]:
    count = viable if form["cn_to_hk_assets"] else 0
    print("  %-6s %-38s → %s"
          % (form["id"], form["name"], required_mechanism(count, form["cn_to_hk_sensitive"], thresholds)))
print()
print("R-B（标准合同）在本项目的适用窗口：")
print("  形态A：需 %d 人以下才落到标准合同，但那样低于可验证下界 %d"
      % (thresholds["sensitive_assessment_at_or_above"], viable))
print("  形态B：需 %d ~ %d 人才落到标准合同，而本项目可用样本远低于下限"
      % (thresholds["non_sensitive_exempt_below"], thresholds["non_sensitive_assessment_at_or_above"]))

在可验证规模 N_eff = 30000 人这一档上，三种形态分别落到：
  形态A    双向 L3（嵌入 hk→cn + 梯度 cn→hk）             → 数据出境安全评估
  形态B    单向 L3（仅嵌入 hk→cn，不回传梯度）+ PSI            → 豁免（无需申报/订立/认证）
  形态C    无任何 cn→hk 个人信息流动                       → 不适用（该方向不提供个人信息）

R-B（标准合同）在本项目的适用窗口：
  形态A：需 10000 人以下才落到标准合同，但那样低于可验证下界 30000
  形态B：需 100000 ~ 1000000 人才落到标准合同，而本项目可用样本远低于下限


## 结论

**R-B（标准合同）在本项目几乎没有适用窗口。** 原因是它被夹在两条阈值之间，而本项目的两个关键数字都落在窗口之外：

| L3 形态 | cn→hk 提供什么 | 在 N_eff = 30,000 这一档 | 为什么 |
|---|---|---|---|
| 形态 A 双向 | 标识符哈希 + **梯度（敏感）** | **安全评估** | 敏感个人信息 ≥1 万人（第七条） |
| 形态 B 单向 | 仅标识符哈希（非敏感） | **豁免** | 非敏感且不满 10 万人（第五条第（四）项） |
| 形态 C 无流动 | 无 | 不适用 | cn→hk 不提供个人信息，PIPL 出境条款不触发 |

**要落到 R-B，形态 A 需人数 <1 万（低于可验证下界），形态 B 需人数 ≥10 万（远超本项目可用样本）。两头都够不着。**

因此本步的实际结论不是「R-B 好不好」，而是：

> **真正的决策不在「选哪条出境路径」，而在「L3 采用哪种形态」。**
> 形态一旦定下，路径就被法条自动决定了，没有选择余地。

形态 B（单向流动）的合规优势是压倒性的——从最重的安全评估直接跳到豁免。
代价在建模侧：不回传梯度意味着被动方的编码器无法随主动方的标签迭代优化，
表示质量会下降多少，须由 M5 用实验回答，而不是在这里推测。